# Melee Combat Theory
Currently, the melee combat logic is contained within `server/src/state/combat.rs`, specifically the `do_attack` function. This function is responsible for the following:
1. Ensure the attacker is allowed to attack the defender
2. Calculate the attacker's and defender's weapon skill value, $a$ and $d$ respectively. This is their modified weapon skill (base + items + spells)
3. Modify $d$ based on the attacker's positioning
4. Modify $d$ based on the defender's state (stunned or not attacking?)

We then take the difference between $a$ and $d$, and use that to determine the attack's success and a bonus value:

$\Delta_{skill} = a - d$

If $\Delta_{skill} < 0$, this mean the defender's skill is higher than the attacker's and the attack becomes less likely to succeed, and may also incur a negative damage bonus. The full mapping from $\Delta_{skill}$ to `chance` and `bonus` is:

| $\Delta_{skill}$ range | `chance` | `bonus` | $P_{hit}$ |
|---|---:|---:|---:|
| $\Delta_{skill} < -40$ | 1 | -16 | 0.05 |
| $-40 \le \Delta_{skill} < -36$ | 2 | -8 | 0.10 |
| $-36 \le \Delta_{skill} < -32$ | 3 | -4 | 0.15 |
| $-32 \le \Delta_{skill} < -28$ | 4 | -2 | 0.20 |
| $-28 \le \Delta_{skill} < -24$ | 5 | -1 | 0.25 |
| $-24 \le \Delta_{skill} < -20$ | 6 | 0 | 0.30 |
| $-20 \le \Delta_{skill} < -16$ | 7 | 0 | 0.35 |
| $-16 \le \Delta_{skill} < -12$ | 8 | 0 | 0.40 |
| $-12 \le \Delta_{skill} < -8$ | 9 | 0 | 0.45 |
| $-8 \le \Delta_{skill} < -4$ | 10 | 0 | 0.50 |
| $-4 \le \Delta_{skill} < 0$ | 11 | 0 | 0.55 |
| $\Delta_{skill} = 0$ | 12 | 0 | 0.60 |
| $0 < \Delta_{skill} < 4$ | 13 | 0 | 0.65 |
| $4 \le \Delta_{skill} < 8$ | 14 | 0 | 0.70 |
| $8 \le \Delta_{skill} < 12$ | 15 | 0 | 0.75 |
| $12 \le \Delta_{skill} < 16$ | 16 | 1 | 0.80 |
| $16 \le \Delta_{skill} < 20$ | 17 | 2 | 0.85 |
| $20 \le \Delta_{skill} < 24$ | 18 | 3 | 0.90 |
| $24 \le \Delta_{skill} < 28$ | 19 | 4 | 0.95 |
| $28 \le \Delta_{skill} < 32$ | 19 | 5 | 0.95 |
| $32 \le \Delta_{skill} < 36$ | 19 | 10 | 0.95 |
| $36 \le \Delta_{skill} < 40$ | 19 | 15 | 0.95 |
| $\Delta_{skill} \ge 40$ | 19 | 20 | 0.95 |

A die is then rolled from 1 to 20, $D_{20}$, and if $D_{20} \le chance$, the attack hits and the damage bonus is applied. Otherwise, the attack misses and no damage is dealt.

---

## Damage Calculation Overview

Once a hit is confirmed (die roll ≤ `chance`, and the Mercenary-line dodge check fails), the engine moves through a two-stage pipeline before hitpoints are actually modified.

### Stage 1 — Base Damage Assembly (`do_attack`)

The raw damage value is built from three additive sources and then adjusted by the skill-differential `bonus`:

| Component | Formula |
|---|---|
| Weapon base | `attacker.weapon + rand(6) + 1` |
| Strength bonus | `rand(attacker.strength / 2)` (only when `strength > 3`) |
| Low-roll crits | die = 2 --> `+rand(6)+1`; die = 1 --> `+rand(6)+rand(6)+2` |
| Skill-diff bonus | `+bonus` (from the $\Delta_{skill}$ table, range −16 to +20) |

For **Surround Hit** (`SK_SURROUND`), secondary AoE targets each receive 75% of the pre-bonus damage (`odam * 3/4`), resolved independently (separate dodge roll, separate `do_hurt` call).

---

### Stage 2 — Damage Application (`do_hurt`)

`do_hurt` takes `(attacker, defender, dam, type_hurt)` where `type_hurt` controls the damage pipeline variant.

#### 2a. Armour wear
If the defender is a player, their equipped armour pieces are degraded in proportion to the incoming `dam` before any other reduction is applied (`item_damage_armor`).

#### 2b. Magical Shield absorption (`SK_MSHIELD`)
For each active `SK_MSHIELD` spell item on the defender:

$$tmp = \left(\frac{active}{1024} + 1\right)$$
$$absorption = (dam + tmp - armor) \times 5$$

If `absorption ≥ active`, the shield is destroyed (item freed). Otherwise the shield's `active` charge is reduced by `absorption`. The armor value is re-read after all shields are resolved.

#### 2c. Damage scaling by type

| `type_hurt` | Used by | Formula |
|---|---|---|
| `0` | Melee, environmental | `(dam − armor) × 250` |
| `1` | Spell damage | `(dam − armor) × 750` |
| `2` | Trap / special events | `(dam − armor) × 750` (no XP, no kill EXP) |
| `3` | Reactive (`gethit`) | `dam × 1000` (armor is **ignored**) |

If `dam ≤ armor` after subtraction (types 0, 1, 2), the result is clamped to 0 — full mitigation. Scaled values are stored and tracked as milli-HP (divide by 1000 to get displayed HP).

**Immortal characters** (`CharacterFlags::Immortal`) always receive `dam = 0` regardless of type.

#### 2d. God Save
When a hit would reduce `a_hp` below 500 **and** the defender's `luck ≥ 100`, a saving throw is attempted:

$$P_{save} = \frac{5000 + luck}{10000}$$

On success the defender's HP is restored to full, `luck` is halved, and the character is teleported to their bound temple. This check is skipped inside arena zones.

#### 2e. HP deduction and death
$$a\_hp \leftarrow a\_hp - dam$$

- HP in the range `[500, 8000)` triggers a low-HP warning.
- HP below 500 --> character is killed, experience is awarded to the attacker (scaled by defender rank, bonus for active protection/enhance/bless spells), and the kill notification chain fires.

#### 2f. Reactive damage (`gethit_dam`)
If the *defender* has a non-zero `gethit_dam` value and the incoming hit was type `0` (melee), the attacker is immediately counter-hit:

$$reactive\_dam = rand(gethit\_dam) + 1$$

This is fed back into `do_hurt` as `type_hurt = 3` (armor-bypassing), so the attacker's own armor cannot reduce it.

A thorough review of the existing character templates revealed that this reactive damage effect is not currently used by any NPCs.
---
